In [7]:
%pip install tensorflow scikit-learn numpy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [17]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

In [41]:
print("Loading data...")

train_x = np.load("data/train_x.npy")
train_y = np.load("data/train_y_encoded.npy")
train_x = train_x / np.max(np.abs(train_x))


val_x = np.load("data/valid_x.npy")
val_y = np.load("data/val_y_encoded.npy")

print("train_x shape:", train_x.shape)
print("train_y shape:", train_y.shape)

Loading data...
train_x shape: (1200, 80000, 1)
train_y shape: (1200,)


In [42]:
assert train_x.shape[0] == train_y.shape[0], "X и Y имеют разное количество samples"

num_classes = len(np.unique(train_y))
print("Classes:", num_classes)


Classes: 20


In [43]:
def make_spectrogram(x):
    # x: (batch, samples, 1)
    x = x[..., 0]  # убираем лишний канал
    spectrogram = tf.signal.stft(
        x,
        frame_length=256,
        frame_step=128
    )
    spectrogram = tf.abs(spectrogram)
    spectrogram = spectrogram[..., tf.newaxis]  # добавляем канал
    return spectrogram


train_x = make_spectrogram(train_x)
val_x = make_spectrogram(val_x)


In [44]:
# X_train, X_val, y_train, y_val = train_test_split(
#     train_x,
#     train_y,
#     test_size=0.2,
#     random_state=42
# )

X_train = train_x
X_val = val_x
y_train = train_y
y_val = val_y


In [45]:
num_classes = len(np.unique(train_y))

model = tf.keras.Sequential([

    layers.Conv2D(16, (3,3), activation="relu", input_shape=X_train.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

Model: "sequential_5"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_6 (Conv2D)           (None, 622, 127, 16)      160       
                                                                 
 batch_normalization_6 (Batc  (None, 622, 127, 16)     64        
 hNormalization)                                                 
                                                                 
 max_pooling2d_6 (MaxPooling  (None, 311, 63, 16)      0         
 2D)                                                             
                                                                 
 conv2d_7 (Conv2D)           (None, 309, 61, 32)       4640      
                                                                 
 batch_normalization_7 (Batc  (None, 309, 61, 32)      128       
 hNormalization)                                                 
                                                      

In [46]:
print("Training...")

history = model.fit(
    X_train,
    y_train,
    epochs=40,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

Training...
Epoch 1/40
38/38 [==============================] - 4s 44ms/step - loss: 12.7288 - accuracy: 0.0533 - val_loss: 6.2553 - val_accuracy: 0.0700
Epoch 2/40
38/38 [==============================] - 1s 38ms/step - loss: 4.0213 - accuracy: 0.0575 - val_loss: 4.5671 - val_accuracy: 0.0650
Epoch 3/40
38/38 [==============================] - 1s 38ms/step - loss: 3.4534 - accuracy: 0.0675 - val_loss: 4.5212 - val_accuracy: 0.0800
Epoch 4/40
38/38 [==============================] - 1s 37ms/step - loss: 3.2852 - accuracy: 0.0725 - val_loss: 5.1029 - val_accuracy: 0.0625
Epoch 5/40
38/38 [==============================] - 1s 38ms/step - loss: 3.6950 - accuracy: 0.0742 - val_loss: 5.2797 - val_accuracy: 0.0650
Epoch 6/40
38/38 [==============================] - 1s 38ms/step - loss: 3.1095 - accuracy: 0.0733 - val_loss: 4.5138 - val_accuracy: 0.0475
Epoch 7/40
38/38 [==============================] - 1s 38ms/step - loss: 3.0787 - accuracy: 0.0733 - val_loss: 3.5725 - val_accuracy: 0.0625


In [70]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt


train_x = np.load("data/train_x.npy")
train_y = np.load("data/train_y_encoded.npy")
val_x = np.load("data/valid_x.npy")
val_y = np.load("data/val_y_encoded.npy")

train_x = train_x / np.max(np.abs(train_x))
val_x = val_x / np.max(np.abs(val_x))

N = 16
unique, counts = np.unique(train_y, return_counts=True)
class_counts = dict(zip(unique, counts))
top_classes = sorted(class_counts, key=class_counts.get, reverse=True)[:N]

train_mask = np.isin(train_y, top_classes)
val_mask = np.isin(val_y, top_classes)

train_x = train_x[train_mask]
train_y = train_y[train_mask]
val_x = val_x[val_mask]
val_y = val_y[val_mask]

class_mapping = {cls:i for i, cls in enumerate(top_classes)}
train_y = np.array([class_mapping[y] for y in train_y])
val_y = np.array([class_mapping[y] for y in val_y])

num_classes = N
print("Используемые классы:", np.unique(train_y))

# -----------------------
# 3. Преобразуем в спектрограммы
# -----------------------
def make_spectrogram(x, target_len=80000):
    """
    x: (batch, samples, 1)
    target_len: приводим все аудио к одной длине
    """
    x = x[..., 0]  # убираем лишний канал
    
    # если аудио короче target_len, дополняем нулями, если длиннее — обрезаем
    x_fixed = np.zeros((x.shape[0], target_len))
    for i in range(x.shape[0]):
        length = min(target_len, x.shape[1])
        x_fixed[i,:length] = x[i,:length]
    
    # STFT
    spectrogram = tf.signal.stft(
        x_fixed,
        frame_length=256,
        frame_step=128
    )
    spectrogram = tf.abs(spectrogram)
    spectrogram = spectrogram[..., tf.newaxis]  # добавляем канал
    return spectrogram.numpy()

train_x = make_spectrogram(train_x)
val_x = make_spectrogram(val_x)

print("train_x shape:", train_x.shape)
print("val_x shape:", val_x.shape)

model = tf.keras.Sequential([
    layers.Conv2D(32, (3,3), activation="relu", input_shape=train_x.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_x, train_y,
    validation_data=(val_x, val_y),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop]
)

Используемые классы: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]
train_x shape: (992, 624, 129, 1)
val_x shape: (328, 624, 129, 1)
Model: "sequential_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_46 (Conv2D)          (None, 622, 127, 32)      320       
                                                                 
 batch_normalization_40 (Bat  (None, 622, 127, 32)     128       
 chNormalization)                                                
                                                                 
 max_pooling2d_36 (MaxPoolin  (None, 311, 63, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_47 (Conv2D)          (None, 309, 61, 64)       18496     
                                                                 
 batch_normalization_41 (Bat  (None, 309, 61, 64

In [71]:
model.save("planet_sound_model.h5")
print("Model saved: planet_sound_model.h5")

Model saved: planet_sound_model.h5
